# 01 · Demand: the network total

**Purpose.** Build the clean 30-minute Ausgrid network total that every later notebook uses, and show the checks that say whether it can be trusted.

**Paper section served.** II Data, and the data-access Appendix.

**Inputs** (raw and read-only; paths come from `config.local.yaml`)

- Ausgrid distribution zone substation data, reporting years FY2015 to FY2026. One CSV per substation, one row per day, 96 columns of 15-minute MW. Source: Ausgrid research data sets.

**Paper artifacts.** None.

**Supporting checks** (`outputs/checks`)

- `check_data_quality.csv`: what the cleaning did to each reporting-year file.
- `check_annual_summary.csv`: each year's minimum, maximum, share of half hours kept and energy.
- `check_kept_by_month.csv`: share of each month kept.
- `check_join_steps.csv`: the level step where reporting years join at 1 May.
- `check_clock.csv`: substations that appear to stay on standard time during daylight saving.
- `check_sign.csv`: the change in sign convention for reverse flow.
- Shown only, not saved: the reader against the stress-test minima (section 4).

**Data for later notebooks** (`data/interim`)

- `network_15min.parquet`: scaled-up network total and reporting share, every 15 minutes, May 2014 to April 2026. Notebook 07 re-thresholds this file.
- `network_30min.parquet`: the 30-minute network total for 2015 to 2025, blank where invalid. This is the series every later notebook reads.

**Main process**

1. Set up.
2. Read and clean every reporting-year file, then join them.
3. Report what the cleaning did to each file.
4. Check the reader against the stress-test minima.
5. Apply the reporting threshold, average to half hours and cut to 2015 to 2025.
6. Check each year: minimum, maximum, share kept, energy.
7. Check each month's share kept.
8. Check the joins at 1 May.
9. Check the clocks around daylight saving.
10. Quantify the change in sign convention.
11. Result.

Times are local clock time, as Ausgrid publishes them. Series are indexed by interval start; tables show the time an interval ends, which matches Ausgrid's column labels.

## 1. Set up

Load the configuration. Every parameter that is a decision (thresholds, run lengths, shares) is read from it, so nothing in this notebook is a magic number.

In [1]:
import pandas as pd

from pynmdf import clean, targets
from pynmdf.config import load_config

cfg = load_config()
interim, checks = cfg["dirs"]["interim"], cfg["dirs"]["checks"]
study, cleaning = cfg["study"], cfg["cleaning"]

pd.set_option("display.width", 220, "display.max_columns", 30, "display.max_rows", 60)
pd.set_option("display.float_format", "{:,.1f}".format)

## 2. Read, clean and join the reporting-year files

Each file is cleaned on its own, because substations are added, retired and renamed between years and a substation's usual share of load is only meaningful within one file. The steps are: put back in time order the days that were published scrambled; correct blocks published in kW; blank outlier days and stuck runs; drop substations with more than 20% of intervals blank or invalid; fill gaps of up to two hours; sum the reporting substations and scale up by their usual share of load.

The result is one unbroken 15-minute series with two columns: the scaled-up total and the share of usual load that reported. The threshold is applied later (section 5), so that notebook 07 can test other thresholds without reading the raw files again. This cell takes a few minutes.

In [2]:
build = clean.build_network_total(cfg)
build.network.to_parquet(interim / "network_15min.parquet")

print(f"{len(build.network):,} intervals, {build.network.index[0]} to {build.network.index[-1]}")

420,768 intervals, 2014-05-01 00:00:00 to 2026-04-30 23:45:00


## 3. What the cleaning did to each file

One column per reporting year. The rows to read first are `substations_kept`, `kept_pct` (share of 15-minute intervals that pass the 95% reporting threshold) and `dropped_load_pct`.

`dropped_load_pct` matters because a dropped substation is not scaled up: its load leaves the total for that whole file. It is the mean load of the dropped substations (blanks counted as zero) as a percentage of the kept substations' mean load, and it shows how large a level step the 20% rule can put between one reporting year and the next.

`scrambled_days_restored` counts substation-days whose 96 values were published in alphabetical order of interval number (1, 10, 11 ... 19, 2, 20 ...) under a time-ordered header. A day is re-ordered only when that makes it much smoother. Every case found is in the FY2023 file, from 1 to 30 April 2023.

In [3]:
quality = build.quality
quality.to_csv(checks / "check_data_quality.csv")

# Shown as objects so that counts print as whole numbers once the table is turned on its side.
quality.drop(columns=["diagnostic_min_mw", "stress_test_min_mw"]).round(1).astype(object).T

reporting_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
substations_read,176,176,220,223,223,223,199,178,194,176,175,175
scrambled_days_restored,0,0,0,0,0,0,0,0,2834,0,0,0
unit_values_corrected,0,119,0,0,0,0,0,0,18109,0,0,0
unit_substations,0,1,0,0,0,0,0,0,7,0,0,0
outlier_substation_days,2,11,4,2,25,0,2,53,20,0,0,0
stuck_values,50686,100525,48454,57851,64812,43841,40288,12668,27120,96,416,116
substations_dropped,13,11,45,44,49,46,24,8,17,2,1,0
dropped_wholly_blank,1,2,36,36,39,42,20,0,13,0,0,0
dropped_load_mw,45.5,22.4,29.8,17.2,7.2,8.3,31.3,52.3,28.2,27.9,8.6,0.0
dropped_load_pct,1.9,0.9,1.2,0.7,0.3,0.3,1.3,2.3,1.2,1.3,0.4,0.0


## 4. Check: the reader against the stress-test minima

The data stress test of 11 November 2026 recorded a 30-minute minimum for each file under a simpler diagnostic rule: the unit fix, the 20% rule, a two-hour fill, and only the intervals where every kept substation reports. Reproducing those numbers shows the files are being read the same way. A gap above about 1% needs explaining.

This is not the final rule. The diagnostic copies the stress test's fill, which also fills the first two hours of longer gaps; the final rule in section 2 fills only gaps of two hours or less.

In [4]:
stress = quality[["diagnostic_min_mw", "stress_test_min_mw"]].copy()
stress["gap_pct"] = 100 * (stress["diagnostic_min_mw"] / stress["stress_test_min_mw"] - 1)
stress.T

reporting_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
diagnostic_min_mw,"1,377.9","1,390.7","1,497.4","1,507.1","1,511.0","1,473.0","1,441.5","1,424.7","1,294.0","1,372.5",941.0,880.1
stress_test_min_mw,"1,378.0","1,391.0","1,497.0","1,507.0","1,511.0","1,473.0","1,442.0","1,425.0","1,294.0","1,364.0",941.0,880.0
gap_pct,-0.0,-0.0,0.0,0.0,0.0,-0.0,-0.0,-0.0,-0.0,0.6,0.0,0.0


## 5. Threshold, half hours and the study window

An interval is kept only when at least 95% of usual load reported. Pairs of 15-minute totals are then averaged, and a half hour is valid only when both of its parts are. Finally the series is cut to calendar years 2015 to 2025.

In [5]:
total = clean.half_hourly_total(
    build.network, cleaning["min_reporting_share"], study["first_year"], study["last_year"]
)
total.to_frame().to_parquet(interim / "network_30min.parquet")

print(f"{len(total):,} half hours, {total.index[0]} to {total.index[-1]}")
print(f"{total.notna().mean():.1%} kept")

192,864 half hours, 2015-01-01 00:00:00 to 2025-12-31 23:30:00


96.4% kept


## 6. Check: each year's minimum, maximum, share kept and energy

Three things to look for. The maximum should sit between about 4,100 and 5,300 MW in every year; a value near 88,000 MW would mean the unit error was not corrected. The share kept shows how much of each year the minimum was searched over. Energy (mean valid MW times the hours in the year) exposes level steps caused by the changing set of substations.

In [6]:
annual = targets.annual_summary(total)
annual.to_csv(checks / "check_annual_summary.csv")

plausible = annual["max_mw"].between(4000, 5400)
print("Annual maximum plausible in every year:", bool(plausible.all()))
annual.assign(kept_pct=100 * annual["kept_share"]).drop(columns="kept_share")

Annual maximum plausible in every year: True


,min_mw,min_time,max_mw,max_time,energy_gwh,kept_pct
year,,,,,,
2015,"1,368.1",2015-04-05 04:30:00,"4,632.8",2015-11-20 17:00:00,"21,444.1",98.4
2016,"1,432.6",2016-04-24 05:00:00,"4,717.1",2016-02-25 17:00:00,"22,012.8",98.8
2017,"1,474.3",2017-04-17 04:30:00,"5,268.9",2017-02-10 17:00:00,"22,386.1",99.3
2018,"1,495.1",2018-04-29 05:00:00,"4,675.8",2018-02-14 17:30:00,"22,291.2",100.0
2019,"1,473.0",2019-11-10 05:00:00,"5,146.2",2019-01-31 17:30:00,"21,931.3",97.7
2020,"1,429.5",2020-04-12 05:00:00,"5,065.7",2020-01-23 18:00:00,"20,926.9",94.8
2021,"1,357.4",2021-10-17 14:30:00,"4,459.0",2021-06-10 18:00:00,"19,805.0",90.9
2022,"1,428.3",2022-04-17 05:00:00,"4,226.3",2022-07-19 18:00:00,"20,174.5",96.7
2023,"1,080.0",2023-10-29 13:00:00,"4,756.6",2023-03-06 18:00:00,"19,286.6",96.3


## 7. Check: share of each month kept

A monthly minimum taken from part of a month may have missed the true one. Months with less than 90% of half hours kept are flagged here; from notebook 05 on they are left out of fitting and scoring.

In [7]:
kept = targets.kept_share_by_month(total)
kept.to_csv(checks / "check_kept_by_month.csv")

(100 * kept).round().astype(int)

month,1,2,3,4,5,6,7,8,9,10,11,12
year,,,,,,,,,,,,
2015,100,89,97,100,99,97,100,100,99,100,100,100
2016,94,100,100,95,100,97,100,100,100,100,100,100
2017,100,100,100,100,97,95,100,100,100,100,100,100
2018,100,100,100,100,100,100,100,100,100,100,100,100
2019,100,100,91,100,86,100,100,96,99,100,100,100
2020,97,100,100,100,100,100,100,100,100,84,57,100
2021,100,100,90,0,100,100,100,100,100,100,100,100
2022,100,100,97,90,100,100,100,100,97,77,100,100
2023,100,100,93,100,100,97,93,93,90,100,97,93


In [8]:
thin = kept.stack()
thin = thin[thin < cfg["targets"]["month_min_kept_share"]]

print(f"{len(thin)} of {kept.size} months are below 90% kept:")
print(", ".join(f"{year}-{month:02d} ({share:.0%})" for (year, month), share in thin.items()))

9 of 132 months are below 90% kept:
2015-02 (89%), 2019-05 (86%), 2020-10 (84%), 2020-11 (57%), 2021-04 (0%), 2022-10 (77%), 2023-09 (90%), 2024-01 (6%), 2024-08 (87%)


## 8. Check: the joins at 1 May

Reporting years meet at midnight on 1 May. `build_network_total` already stops with an error unless the files join into one unbroken 15-minute series, with no interval repeated or missing.

An unbroken index does not guarantee an unbroken level, so the table measures the step at each join. `change_mw` is the first hour of 1 May minus the last hour of 30 April; `usual_change_mw` is the median of the same change over the seven nights either side; `step_mw` is the difference. `rank` places that step among every night of the series (1 is the largest upward step), so a low rank means the two files do not meet at the same level.

The 2021 and 2025 joins cannot be measured: April 2021 has no valid data, and 30 April 2025 is absent from the FY2025 file.

In [9]:
unbroken = pd.date_range(total.index[0], total.index[-1], freq="30min")
print("Index unique and unbroken:", total.index.is_unique and total.index.equals(unbroken))

steps = clean.join_steps(total)
steps.to_csv(checks / "check_join_steps.csv")
steps.T

Index unique and unbroken: True


join_at_1_may,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
change_mw,-66.8,-15.8,-65.5,-76.7,-122.6,-124.4,NaN,47.1,-199.8,-43.7,NaN
usual_change_mw,-133.2,-114.4,-112.7,-112.8,-112.0,-100.9,NaN,-97.7,-115.4,-109.9,-114.0
step_mw,66.4,98.6,47.2,36.1,-10.7,-23.4,NaN,144.8,-84.4,66.2,NaN
step_pct,3.0,4.8,2.3,1.6,-0.5,-1.1,NaN,7.2,-4.1,3.2,NaN
rank,30.0,12.0,81.0,169.0,"2,804.0","3,302.0",NaN,6.0,"3,778.0",31.0,NaN
nights_ranked,"3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0","3,827.0"


## 9. Check: clocks around daylight saving

Ausgrid states that times are local clock time, with daylight saving in summer. This check tests that, substation by substation. People follow the clock, so the morning rise in demand should sit at the same clock time before and after a clock change. For each substation above 3 MW, the slot of the steepest weekday rise between 03:00 and 09:30 is found over the three weeks before and the three weeks after each change.

The table gives the share of load whose morning rise moves by 45 minutes or more. A large share moving **earlier in October** (when daylight saving starts), and **later in April** (when it ends), means those substations are being published in standard time, one hour behind the rest through summer. The test is indirect: ordinary seasonal drift is one or two 15-minute slots, which is why a few per cent appears in every year. The data are not corrected; this is a stated limitation.

In [10]:
clocks = build.clocks
clocks.to_csv(checks / "check_clock.csv")

clocks.T

reporting_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
october_earlier_pct,0.0,0.4,0.0,1.2,0.6,0.5,0.2,0.1,0.3,21.7,12.8,10.0
october_later_pct,4.9,11.0,5.8,10.6,4.7,8.3,12.0,25.5,6.8,2.6,2.6,7.7
april_earlier_pct,2.4,5.4,7.3,2.9,5.3,6.9,7.1,8.0,17.0,6.3,1.5,13.0
april_later_pct,6.3,5.9,2.2,9.8,5.0,3.6,1.7,1.6,2.0,19.3,5.9,1.5


## 10. Check: the sign convention across files

The analysis takes signs as published (a locked decision), and this table puts a number on the limitation. Ausgrid's notes say many sites measure amps only, so their MW carries no direction.

The check takes the substations whose mean is negative at the reference year's 200 lowest daytime intervals (FY2024, the first file in which reverse flow is clearly signed) and follows the same substations through every file. `reference_substations_mw` is their summed mean at each file's own lowest daytime intervals. A positive sum that turns negative between two adjacent files, while `substations_with_negatives` jumps, is the signature of reverse flow that was recorded unsigned in the earlier file.

In [11]:
sign = clean.sign_check(build, cfg["checks"]["sign_reference_year"])
sign.to_csv(checks / "check_sign.csv")

sign.T

reporting_year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
substations_with_negatives,1.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,3.0,43.0,47.0,68.0
reference_substations_found,23.0,24.0,23.0,24.0,25.0,25.0,25.0,23.0,25.0,25.0,25.0,25.0
reference_substations_mw,102.3,109.0,100.8,97.6,87.8,69.8,60.5,40.9,46.5,-50.2,-77.5,-90.0
all_negative_mw,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-50.2,-84.8,-112.9


## 11. Result

The network total is built and saved. The table below is the headline of this notebook: each calendar year's minimum, when it fell, and how much of the year was searched.

In [12]:
result = annual[["min_mw", "min_time"]].assign(
    weekday=annual["min_time"].sub(targets.HALF_HOUR).dt.day_name(),
    kept_pct=100 * annual["kept_share"],
)
kept_subs = quality["substations_kept"]
lost = quality["dropped_load_pct"]
large_steps = (steps["step_pct"].abs() > 2).sum()

print(f"Substations kept per file: {kept_subs.min()} to {kept_subs.max()}")
print(f"Load lost to the 20% rule: {lost.min():.1f}% to {lost.max():.1f}% by file")
print(f"Scrambled substation-days restored: {quality['scrambled_days_restored'].sum():,}")
print(f"Largest gap to the stress-test minima: {stress['gap_pct'].abs().max():.1f}%")
print(f"Months below 90% kept: {len(thin)} of {kept.size}")
print(f"Joins with a level step above 2%: {large_steps} of {steps['step_mw'].count()} measurable")
earlier = clocks["october_earlier_pct"]
worst = earlier.idxmax()
print(f"Largest share of load moving earlier in October: {earlier[worst]:.0f}% (FY{worst})")
result

Substations kept per file: 163 to 179
Load lost to the 20% rule: 0.0% to 2.3% by file
Scrambled substation-days restored: 2,834
Largest gap to the stress-test minima: 0.6%
Months below 90% kept: 9 of 132
Joins with a level step above 2%: 6 of 9 measurable
Largest share of load moving earlier in October: 22% (FY2024)


,min_mw,min_time,weekday,kept_pct
year,,,,
2015,"1,368.1",2015-04-05 04:30:00,Sunday,98.4
2016,"1,432.6",2016-04-24 05:00:00,Sunday,98.8
2017,"1,474.3",2017-04-17 04:30:00,Monday,99.3
2018,"1,495.1",2018-04-29 05:00:00,Sunday,100.0
2019,"1,473.0",2019-11-10 05:00:00,Sunday,97.7
2020,"1,429.5",2020-04-12 05:00:00,Sunday,94.8
2021,"1,357.4",2021-10-17 14:30:00,Sunday,90.9
2022,"1,428.3",2022-04-17 05:00:00,Sunday,96.7
2023,"1,080.0",2023-10-29 13:00:00,Sunday,96.3
